# Module 3 — Multilingual Personalised Tutor

Takes the gap analysis from Module 2 and generates a full remedial lesson in the student's regional language.

Prerequisite: Run Module 2 and save `diagnostic_report_*.json`.

In [ ]:
!pip install google-generativeai -q

In [ ]:
import google.generativeai as genai
import json

GEMINI_API_KEY = ""
MODEL_NAME     = "gemini-2.0-flash"

genai.configure(api_key=GEMINI_API_KEY)

In [ ]:
with open('diagnostic_report_fractions.json', 'r', encoding='utf-8') as f:
    gap_json = json.load(f)

print(gap_json['mastery_level'])
print(gap_json['overall_score'])
print(len(gap_json['identified_gaps']))

In [ ]:
def build_tutor_prompt(regional_language, grade_level, gap_json, learning_path):
    gaps_text = '\n\n'.join(
        f"Gap {i+1}: {g['micro_skill']}\n"
        f"  Misconception: {g['misconception']}\n"
        f"  Root Cause: {g['root_cause']}\n"
        f"  Severity: {g['severity']}\n"
        f"  Priority: {g['recommended_priority']}"
        for i, g in enumerate(gap_json['identified_gaps'])
    )

    path_text = '\n'.join(f"{i+1}. {s}" for i, s in enumerate(learning_path))

    return f"""You are an expert multilingual tutor specializing in personalized remedial education for K-12 students.

Your explanations should feel like a patient teacher sitting beside the student.

Regional Language: {regional_language}
Student Grade: {grade_level}
Mastery Level: {gap_json['mastery_level']}
Overall Score: {gap_json['overall_score']}/100

Gap Analysis:
{gaps_text}

Learning Path:
{path_text}

OBJECTIVES:
1. Explain the concept from scratch.
2. Assume zero prior knowledge.
3. Use extremely simple language.
4. Use examples from the student's daily life.
5. Avoid technical vocabulary unless necessary.
6. Increase confidence through positive reinforcement.

TEACHING METHOD:
Step 1: Explain the idea
Step 2: Give a visual mental model
Step 3: Use a real-life analogy
Step 4: Solve one example
Step 5: Explain common mistakes
Step 6: Give guided practice
Step 7: Give independent practice

REAL LIFE EXAMPLES — prefer: local markets, cricket, school bags, fruits, money, cooking, family, travel, shops, festivals.

PRACTICE:
- 3 guided questions (easy, with scaffolding)
- 2 independent/medium questions
- 1 challenge question
Each must include: question, hint, solution, why_this_works.

LANGUAGE: ENTIRE output must be in {regional_language}. No English except required math/science terms.

Return ONLY valid JSON. No markdown. No explanations outside JSON.

{{
  "simplified_explanation": "",
  "visual_mental_model": "",
  "real_life_analogy": "",
  "worked_example": "",
  "common_mistakes": "",
  "guided_practice": [
    {{"question": "", "hint": "", "solution": "", "why_this_works": ""}},
    {{"question": "", "hint": "", "solution": "", "why_this_works": ""}},
    {{"question": "", "hint": "", "solution": "", "why_this_works": ""}}
  ],
  "independent_practice": [
    {{"question": "", "hint": "", "solution": "", "why_this_works": ""}},
    {{"question": "", "hint": "", "solution": "", "why_this_works": ""}}
  ],
  "challenge_question": {{"question": "", "hint": "", "solution": "", "why_this_works": ""}},
  "revision_checklist": [],
  "motivational_message": "",
  "mnemonic": ""
}}"""

In [ ]:
def generate_lesson(regional_language, grade_level, gap_json, learning_path):
    model = genai.GenerativeModel(
        model_name=MODEL_NAME,
        generation_config={
            'response_mime_type': 'application/json',
            'temperature': 0.4,
            'max_output_tokens': 8192,
        }
    )
    prompt = build_tutor_prompt(regional_language, grade_level, gap_json, learning_path)
    response = model.generate_content(prompt)
    return json.loads(response.text)

In [ ]:
lesson = generate_lesson(
    regional_language = 'Hindi',
    grade_level       = 'Grade 5',
    gap_json          = gap_json,
    learning_path     = gap_json['learning_path'],
)

print(type(lesson))
print(list(lesson.keys()))

In [ ]:
print(lesson['simplified_explanation'])
print(lesson['visual_mental_model'])
print(lesson['real_life_analogy'])
print(lesson['worked_example'])
print(lesson['common_mistakes'])

print('GUIDED PRACTICE')
for i, q in enumerate(lesson['guided_practice'], 1):
    print(i, q['question'])
    print(q['hint'])
    print(q['solution'])
    print(q['why_this_works'])

print('INDEPENDENT PRACTICE')
for i, q in enumerate(lesson['independent_practice'], 1):
    print(i, q['question'])
    print(q['hint'])
    print(q['solution'])

print('CHALLENGE')
c = lesson['challenge_question']
print(c['question'])
print(c['hint'])
print(c['solution'])

print('CHECKLIST')
for item in lesson['revision_checklist']:
    print(item)

print(lesson['motivational_message'])

if lesson.get('mnemonic'):
    print(lesson['mnemonic'])

In [ ]:
out_file = 'lesson_output.json'
with open(out_file, 'w', encoding='utf-8') as f:
    json.dump(lesson, f, indent=2, ensure_ascii=False)